# Clean Federal Reserve Press Releases for RAG

Notebook นี้ทำงานกับ `fed_press_releases.jsonl` โดยไม่แก้ไขไฟล์ต้นฉบับ

สิ่งที่ทำ:
1. แยกหน้า Archive ออกจากข่าวรายฉบับ
2. ตัดข้อความส่วนเกินที่ตรวจจับได้ และซ่อม mojibake แบบ conservative
3. สกัด `publication_date` จากเนื้อหา หรือใช้ URL เป็น fallback
4. สร้าง Clean Master JSONL, RAG chunks, รายการตรวจสอบ และรายงานคุณภาพ

> **ก่อนรัน:** แก้ `INPUT_PATH` ใน Cell 2 ให้ตรงกับตำแหน่งไฟล์บนเครื่องของคุณ ตรวจตัวอย่างผลลัพธ์ก่อนนำเข้า Vector Database เสมอ


In [3]:
from pathlib import Path
from datetime import datetime
from collections import Counter
import json
import re
import hashlib
import pandas as pd


In [4]:
# ===== CONFIG: แก้ path ให้ตรงกับเครื่องของคุณ =====
INPUT_PATH = Path(r"D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases.jsonl")

# Output folder จะถูกสร้างไว้ข้างไฟล์ input
OUTPUT_DIR = INPUT_PATH.parent / "fed_press_releases_cleaned"

CHUNK_SIZE = 1400
CHUNK_OVERLAP = 180

print("Input:", INPUT_PATH.resolve())
print("Input exists:", INPUT_PATH.exists())
print("Output:", OUTPUT_DIR.resolve())

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        "ไม่พบไฟล์ input กรุณาแก้ INPUT_PATH ใน Cell นี้ให้ตรงกับไฟล์ fed_press_releases.jsonl"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


Input: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases.jsonl
Input exists: True
Output: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned


In [5]:
# โหลด JSONL และแยกหน้า Archive ออกจากข่าวรายฉบับ
with INPUT_PATH.open("r", encoding="utf-8-sig") as f:
    raw_records = [json.loads(line) for line in f if line.strip()]

article_records = [r for r in raw_records if r.get("archive_type")]
archive_records = [r for r in raw_records if not r.get("archive_type")]

seen_urls = set()
articles_unique = []
duplicates_removed = 0

for record in article_records:
    url = (record.get("url") or "").strip()
    if url and url in seen_urls:
        duplicates_removed += 1
        continue
    if url:
        seen_urls.add(url)
    articles_unique.append(record)

print("Total raw records:", len(raw_records))
print("Article records:", len(article_records))
print("Archive records:", len(archive_records))
print("Duplicate article URLs removed:", duplicates_removed)

display(pd.DataFrame([
    {"record_type": "articles_raw", "count": len(article_records)},
    {"record_type": "archive_pages", "count": len(archive_records)},
    {"record_type": "unique_articles", "count": len(articles_unique)},
]))


Total raw records: 4658
Article records: 4629
Archive records: 29
Duplicate article URLs removed: 0


,record_type,count
0,articles_raw,4629
1,archive_pages,29
2,unique_articles,4629


In [6]:
# ฟังก์ชันซ่อม Encoding และสกัดวันที่
MOJIBAKE_SEQUENCE_RE = re.compile(r"[\u00C2-\u00F4][\u0080-\u00BF]{1,3}")
RELEASE_RE = re.compile(r"^\s*For\s+(?:immediate\s+release|release\s+at\b.*)$", re.I)
DATE_FORMATS = ("%B %d, %Y", "%b %d, %Y")


def repair_text(text):
    """ซ่อม UTF-8 ที่ถูก decode เป็น Latin-1 แบบระมัดระวัง"""
    if not isinstance(text, str):
        text = str(text or "")
    replacements = 0

    def replace_match(match):
        nonlocal replacements
        try:
            fixed = match.group(0).encode("latin-1").decode("utf-8")
            replacements += 1
            return fixed
        except (UnicodeEncodeError, UnicodeDecodeError):
            return match.group(0)

    text = MOJIBAKE_SEQUENCE_RE.sub(replace_match, text)
    text = text.replace("\xa0", " ").replace("\u200b", "")
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]", "", text)
    return text, replacements


def parse_date_line(value):
    value = (value or "").strip()
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            pass
    return None


def extract_publication_date(raw_content, url):
    lines = (raw_content or "").replace("\r\n", "\n").replace("\r", "\n").split("\n")
    for line in lines[:12]:
        date_value = parse_date_line(line)
        if date_value:
            return date_value.isoformat(), "content"

    # Fallback สำหรับ URL ที่มีวันที่ YYYYMMDD เช่น monetary20090504a.htm
    match = re.search(r"((?:19|20)\d{2})(\d{2})(\d{2})[a-z]?\.htm$", url or "", re.I)
    if match:
        try:
            date_value = datetime.strptime("".join(match.groups()), "%Y%m%d").date()
            return date_value.isoformat(), "url_fallback"
        except ValueError:
            pass
    return None, "not_found"


def normalize_key(value):
    return re.sub(r"\s+", " ", (value or "")).strip().casefold()


def clean_body(raw_content, title):
    text, encoding_repairs = repair_text(raw_content)
    lines = text.replace("\r\n", "\n").replace("\r", "\n").split("\n")
    lines = [line.rstrip() for line in lines]

    release_time_text = None
    for line in lines[:15]:
        if RELEASE_RE.match(line):
            release_time_text = line.strip()
            break

    cleaning_flags = []

    # ตัด contact block ที่อาจอยู่หลัง Share ในข่าวรุ่นเก่า
    share_idx = next(
        (i for i, line in enumerate(lines[:15]) if line.strip().casefold() == "share"),
        None,
    )
    if share_idx is not None:
        contact_window = lines[share_idx + 1:share_idx + 9]
        has_contact_block = any(
            "spokesman" in line.casefold()
            or re.search(r"\(?\d{3}\)?[ .-]*\d{3}[.-]\d{4}", line)
            or ("email" in line.casefold() and ("protected" in line.casefold() or "@" in line))
            for line in contact_window
        )
        if has_contact_block:
            body_start = None
            for j in range(share_idx + 1, min(len(lines), share_idx + 18)):
                candidate = lines[j].strip()
                if len(candidate) >= 80 or ("--" in candidate and len(candidate) >= 40):
                    body_start = j
                    break
            if body_start is not None:
                lines = lines[:share_idx] + lines[body_start:]
                cleaning_flags.append("leading_contact_block_removed")

    fixed_title, _ = repair_text(title)
    title_key = normalize_key(fixed_title)
    result = []

    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.casefold() in {"share", "print"}:
            continue
        if i < 12 and parse_date_line(stripped):
            continue
        if i < 12 and title_key and normalize_key(stripped) == title_key:
            continue
        if i < 15 and RELEASE_RE.match(stripped):
            continue
        if i < 5 and normalize_key(stripped) == "financial services policy committee":
            continue
        result.append(stripped)

    # ตัด media-contact footer เฉพาะเมื่ออยู่ท้ายเอกสาร
    contact_idx = next(
        (
            i for i, line in enumerate(result)
            if i >= max(0, len(result) - 12)
            and re.match(r"^for media inquiries\b", line.strip(), re.I)
        ),
        None,
    )
    if contact_idx is not None:
        result = result[:contact_idx]
        cleaning_flags.append("terminal_media_contact_removed")

    normalized = []
    previous_blank = False
    for line in result:
        stripped = line.strip()
        if not stripped:
            if normalized and not previous_blank:
                normalized.append("")
            previous_blank = True
        else:
            normalized.append(re.sub(r"[ \t]+", " ", stripped))
            previous_blank = False

    cleaned = "\n".join(normalized).strip()
    if not cleaned:
        cleaned = text.strip()
        cleaning_flags.append("body_cleanup_fallback")

    return cleaned, encoding_repairs, release_time_text, cleaning_flags


In [7]:
# Clean ข่าวรายฉบับและจัดเก็บ metadata
cleaned_records = []

for record in articles_unique:
    url = record.get("url") or ""
    publication_date, date_source = extract_publication_date(
        record.get("content", ""), url
    )
    title, _ = repair_text(record.get("title", ""))
    content, encoding_repairs, release_time_text, flags = clean_body(
        record.get("content", ""), title
    )

    if date_source == "url_fallback":
        flags.append("publication_date_from_url")
    elif date_source == "not_found":
        flags.append("publication_date_not_found")

    cleaned_records.append({
        "document_id": hashlib.sha1((url or title).encode("utf-8")).hexdigest(),
        "source": record.get("source") or "Federal Reserve Board",
        "title": title.strip(),
        "publication_date": publication_date,
        "publication_date_source": date_source,
        "release_time_text": release_time_text,
        "document_type": record.get("archive_type") or "unknown",
        "source_archive_year": record.get("archive_year"),
        "url": url,
        "content": content,
        "collected_at": record.get("collected_at"),
        "encoding_repairs": encoding_repairs,
        "cleaning_flags": flags,
    })

print("Cleaned article records:", len(cleaned_records))
display(pd.DataFrame(cleaned_records).head(3)[
    ["title", "publication_date", "publication_date_source", "document_type", "url"]
])


Cleaned article records: 4629


,title,publication_date,publication_date_source,document_type,url
0,Annual notice of asset-size exemption threshol...,2006-12-29,content,general,https://www.federalreserve.gov/newsevents/pres...
1,"Board will be closed on January 2, 2007, in ob...",2006-12-28,content,general,https://www.federalreserve.gov/newsevents/pres...
2,Agencies Release Annual CRA Asset-Size Thresho...,2006-12-27,content,general,https://www.federalreserve.gov/newsevents/pres...


In [9]:
# บันทึก Clean Master, Archive แยกต่างหาก และรายการที่ต้องตรวจ

def write_jsonl(path, records):
    with path.open("w", encoding="utf-8", newline="\n") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")



clean_path = OUTPUT_DIR / "fed_press_releases_clean.jsonl"
archive_path = OUTPUT_DIR / "fed_press_release_archive_pages_raw.jsonl"
review_path = OUTPUT_DIR / "fed_press_releases_needs_review.jsonl"

write_jsonl(clean_path, cleaned_records)
write_jsonl(archive_path, archive_records)

needs_review = [
    r for r in cleaned_records
    if not r["publication_date"] or len(r["content"].strip()) < 40
]
write_jsonl(review_path, needs_review)

print("Clean dataset:", clean_path.resolve())
print("Archive records:", archive_path.resolve())
print("Needs review:", review_path.resolve(), "| records:", len(needs_review))


Clean dataset: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_clean.jsonl
Archive records: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_release_archive_pages_raw.jsonl
Needs review: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_needs_review.jsonl | records: 0


In [10]:
# แบ่งเนื้อหาเป็น chunks สำหรับ RAG
def chunk_text(text, max_chars=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = (text or "").strip()
    if not text:
        return []

    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = min(start + max_chars, text_length)

        if end < text_length:
            lower_bound = start + int(max_chars * 0.65)
            candidates = [
                text.rfind("\\n", lower_bound, end),
                text.rfind(". ", lower_bound, end),
                text.rfind(" ", lower_bound, end),
            ]
            split_at = max(candidates)
            if split_at > start:
                end = split_at + 1 if text[split_at:split_at + 2] == ". " else split_at

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        if end >= text_length:
            break

        start = max(end - overlap, start + 1)

    return chunks


rag_chunks = []
for doc in cleaned_records:
    parts = chunk_text(doc["content"])
    for index, part in enumerate(parts):
        rag_chunks.append({
            "chunk_id": hashlib.sha1(
                f"{doc['document_id']}:{index}".encode("utf-8")
            ).hexdigest(),
            "document_id": doc["document_id"],
            "chunk_index": index,
            "chunk_count": len(parts),
            "source": doc["source"],
            "title": doc["title"],
            "publication_date": doc["publication_date"],
            "release_time_text": doc["release_time_text"],
            "document_type": doc["document_type"],
            "url": doc["url"],
            "content": part,
            "text_for_embedding": (
                f"Title: {doc['title']}\\n"
                f"Publication date: {doc['publication_date'] or 'unknown'}\\n\\n"
                f"{part}"
            ),
        })

chunks_path = OUTPUT_DIR / "fed_press_releases_rag_chunks.jsonl"
write_jsonl(chunks_path, rag_chunks)

print("RAG chunks:", len(rag_chunks))
print("Saved:", chunks_path.resolve())


RAG chunks: 7200
Saved: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_rag_chunks.jsonl


In [11]:
# รายงานคุณภาพข้อมูลและตรวจตัวอย่างก่อนนำเข้า Vector Database
report = {
    "input_file": str(INPUT_PATH.resolve()),
    "input_records": len(raw_records),
    "archive_index_records_separated": len(archive_records),
    "article_records_raw": len(article_records),
    "duplicate_article_urls_removed": duplicates_removed,
    "clean_article_records": len(cleaned_records),
    "rag_chunks": len(rag_chunks),
    "missing_publication_date": sum(not r["publication_date"] for r in cleaned_records),
    "publication_date_source_counts": dict(
        Counter(r["publication_date_source"] for r in cleaned_records)
    ),
    "empty_or_short_clean_content": sum(
        len(r["content"].strip()) < 40 for r in cleaned_records
    ),
    "records_with_encoding_repairs": sum(
        r["encoding_repairs"] > 0 for r in cleaned_records
    ),
    "encoding_sequences_repaired": sum(
        r["encoding_repairs"] for r in cleaned_records
    ),
    "records_with_cleaning_flags": sum(
        bool(r["cleaning_flags"]) for r in cleaned_records
    ),
    "needs_review_records": len(needs_review),
    "outputs": {
        "clean_dataset": str(clean_path.resolve()),
        "archive_pages": str(archive_path.resolve()),
        "rag_chunks": str(chunks_path.resolve()),
        "needs_review": str(review_path.resolve()),
    },
}

report_path = OUTPUT_DIR / "cleaning_report.json"
report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

display(pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in report.items()
    if not isinstance(value, dict)
]))

print("Report:", report_path.resolve())

# Preview: ตรวจเนื้อหาจริงก่อนสร้าง embeddings
for i, doc in enumerate(cleaned_records[:3], start=1):
    print(f"\n--- SAMPLE {i} ---")
    print("Title:", doc["title"])
    print("Publication date:", doc["publication_date"])
    print("URL:", doc["url"])
    print("Content preview:", doc["content"][:800])


,metric,value
0,input_file,D:\Project_Github\gold assisstant\dataset\news...
1,input_records,4658
2,archive_index_records_separated,29
3,article_records_raw,4629
4,duplicate_article_urls_removed,0
5,clean_article_records,4629
6,rag_chunks,7200
7,missing_publication_date,0
8,empty_or_short_clean_content,0
9,records_with_encoding_repairs,511


Report: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\cleaning_report.json

--- SAMPLE 1 ---
Title: Annual notice of asset-size exemption threshold for depository institutions, Regulation C
Publication date: 2006-12-29
URL: https://www.federalreserve.gov/newsevents/pressreleases/bcreg20061229a.htm
Content preview: The Federal Reserve Board on Friday published its annual notice of the asset-size exemption threshold for depository institutions under Regulation C, which implements the Home Mortgage Disclosure Act (HMDA).
The asset-size exemption for depository institutions will increase from $35 million to $36 million based on the annual percentage change in the Consumer Price Index for Urban Wage Earners and Clerical Workers for the twelve-month period ending in November 2006. As a result, depository institutions with assets of $36 million or less as of December 31, 2006, are exempt from collecting data in 2007. An institution’s exemption from collecting d